# 🐖 Notebook 14 | Calibración probabilística y diagramas de confiabilidad

**RECURSO: CPU — no uses GPU.** Sin inferencias ni nuevos entrenamientos.

**Pregunta:** ¿Las puntuaciones de confianza de las detecciones de YOLO11n-seg y Mask R-CNN se corresponden con su frecuencia real de acierto?

- Calibrador **Platt/logístico** predefinido para cada modelo usando **solo las 196 imágenes de validación** del Notebook 13.
- Una detección es correcta si su **máscara** se empareja uno a uno con una máscara anotada a **IoU ≥ 0,50**. Duplicados y detecciones sin emparejar se consideran falsas.
- Evaluación de calibración **únicamente con predicciones del test guardadas por Notebook 09**, sin recalcular máscaras ni retocar configuraciones.
- Figuras profesionales **2×2 antes/después, YOLO/Mask R-CNN**, PNG 600 dpi, PDF vectorial y SVG, sin números inventados.

> Limitaciones: ECE y Brier se definen **entre las detecciones propuestas**, no entre todos los posibles animales; falsos negativos sin detección quedan fuera. El test ya ha sido consultado en las fases anteriores, por tanto esta evaluación es **post hoc/exploratoria** y no valida transferencia externa a otra granja. Las predicciones de una misma fecha no son independientes. Los calibradores son monotónicos y no alteran por sí solos el número de detecciones ni los errores de conteo a umbral fijo.

## A. Instalar dependencias y conectar Drive (CPU)

In [ ]:
%pip -q install "pycocotools>=2.0.8,<3" "scikit-learn>=1.4,<2" "pandas>=2,<3" openpyxl matplotlib pillow


In [ ]:
from google.colab import drive,files as colab_files
from pathlib import Path
from datetime import datetime,timezone
from collections import defaultdict
import json,hashlib,zipfile,math,io
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from matplotlib.patches import Rectangle,Patch
from matplotlib.lines import Line2D
from pycocotools import mask as maskutils
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score,average_precision_score

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
REPORTS=ROOT/'03_Resultados'/'05_Reportes'
METRICS=ROOT/'03_Resultados'/'03_Metricas'
FIGURES=ROOT/'03_Resultados'/'04_Figuras'
EXPORT=ROOT/'05_Exportaciones_ZIP'
for d in [REPORTS,METRICS,FIGURES,EXPORT]:d.mkdir(parents=True,exist_ok=True)
D13=REPORTS/'CERDOS_13_Calibracion_Validacion'
D09=REPORTS/'CERDOS_09_COCO_Armonizado'
P13=D13/'CERDOS_13_Protocolo.json';P09=D09/'CERDOS_09_Protocolo.json'
VGT=D13/'CERDOS_13_GT_val_coco.json';TGT=D09/'CERDOS_09_GT_test_coco.json'
VP={'YOLO11n-seg':D13/'CERDOS_13_Pred_YOLO_val_coco.json',
    'Mask R-CNN':D13/'CERDOS_13_Pred_MaskRCNN_val_coco.json'}
TP={'YOLO11n-seg':D09/'CERDOS_09_Pred_YOLO_coco.json',
    'Mask R-CNN':D09/'CERDOS_09_Pred_MaskRCNN_coco.json'}
for p in [P13,P09,VGT,TGT,*VP.values(),*TP.values()]:
    if not p.is_file():raise FileNotFoundError(f'Falta archivo previo 09 o 13: {p}')
p13=json.loads(P13.read_text(encoding='utf-8'))
p09=json.loads(P09.read_text(encoding='utf-8'))
for a,b in [('yaml_sha256','yaml_sha256'),('yolo_sha256','yolo_sha256'),('mask_sha256','mask_sha256')]:
    assert p13[a]==p09[b],f'Pesos o split distintos: {a}'
for key in ['max_dets','yolo_imgsz','yolo_conf','yolo_iou_nms','mask_score_thresh','mask_bin_thresh']:
    assert p13[key]==p09[key],f'Incompatibilidad de inferencia 09/13: {key}'
assert p13['retina_masks']==p09['yolo_retina_masks']

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):h.update(b)
    return h.hexdigest()

val_gt=json.loads(VGT.read_text(encoding='utf-8'))
test_gt=json.loads(TGT.read_text(encoding='utf-8'))
assert len(val_gt['images'])==196 and len(val_gt['annotations'])==1995
assert len(test_gt['images'])==373 and len(test_gt['annotations'])==4648
assert len({x['file_name'][:10] for x in val_gt['images']})==4
assert len({x['file_name'][:10] for x in test_gt['images']})==5
assert set(x['file_name'][:10] for x in val_gt['images']).isdisjoint(set(x['file_name'][:10] for x in test_gt['images']))
val_preds={m:json.loads(f.read_text(encoding='utf-8')) for m,f in VP.items()}
test_preds={m:json.loads(f.read_text(encoding='utf-8')) for m,f in TP.items()}
print('✅ Val:',len(val_gt['images']),'| Test:',len(test_gt['images']))
print('✅ Pesos y protocolos concuerdan; modelos NO se reentrenan.')


## B. Construir etiquetas binarias con emparejamiento de máscaras (IoU ≥0,50)

In [ ]:
IOU_MIN=0.50
N_BINS=10
SEED=2026
# Considerar todas las predicciones guardadas (confianza de inferencia >=0,001),
# sin filtrar selectivamente el test para mejorar el ECE.
THRESHOLD_SOURCE=float(p13['yolo_conf'])
assert THRESHOLD_SOURCE==float(p13['mask_score_thresh'])==0.001

def det_labels(gt,preds,part,model):
    images={int(x['id']):x for x in gt['images']}
    g_by_image=defaultdict(list)
    for a in gt['annotations']:
        assert int(a['category_id'])==1
        g_by_image[int(a['image_id'])].append(a['segmentation'])
    p_by_image=defaultdict(list)
    for r in preds:
        iid=int(r['image_id'])
        assert iid in images and int(r['category_id'])==1
        assert 0<=float(r['score'])<=1
        p_by_image[iid].append(r)
    records=[];summary=[]
    for iid in sorted(images):
        gt_rle=g_by_image[iid]
        rows=sorted(p_by_image[iid],key=lambda x:float(x['score']),reverse=True)
        # pycocotools calcula IoU directamente desde las máscaras RLE,
        # sin reconstruir arrays grandes para todas las imágenes a la vez.
        if gt_rle and rows:
            matrix=maskutils.iou([r['segmentation'] for r in rows],gt_rle,[0]*len(gt_rle))
        else:
            matrix=np.zeros((len(rows),len(gt_rle)),dtype=np.float64)
        used=set();tp=0
        for j,r in enumerate(rows):
            candidates=[(float(matrix[j,k]),k) for k in range(len(gt_rle)) if k not in used]
            best=max(candidates,default=(0.0,-1))
            ok=int(best[0]>=IOU_MIN)
            if ok:used.add(best[1]);tp+=1
            records.append({'modelo':model,'particion':part,'imagen_id':iid,
                            'imagen':images[iid]['file_name'],'fecha':images[iid]['file_name'][:10],
                            'score_raw':float(r['score']),'correcta_IoU05':ok,
                            'iou_max_libre':best[0]})
        summary.append({'modelo':model,'particion':part,'imagen_id':iid,
                        'fecha':images[iid]['file_name'][:10],
                        'gt':len(gt_rle),'n_pred':len(rows),'TP':tp,
                        'FP':len(rows)-tp,'FN':len(gt_rle)-tp})
    out=pd.DataFrame(records)
    if out.empty:raise RuntimeError(f'Sin predicciones {part} {model}')
    assert len(out)==len(preds) and out['correcta_IoU05'].isin([0,1]).all()
    return out,pd.DataFrame(summary)

labels={};counts=[]
for model in VP:
    a,v_count=det_labels(val_gt,val_preds[model],'validacion',model)
    b,t_count=det_labels(test_gt,test_preds[model],'prueba',model)
    assert len(v_count)==196 and len(t_count)==373
    labels[model]={'val':a,'test':b}
    counts.extend([v_count,t_count])
    print('✅',model,'| val detecciones:',len(a),'| test detecciones:',len(b),
          '| test TP:',int(t_count['TP'].sum()),'FP:',int(t_count['FP'].sum()),'FN:',int(t_count['FN'].sum()))
df_counts=pd.concat(counts,ignore_index=True)


## C. Ajustar Platt/logístico **solo en validación** y evaluar en test

In [ ]:
def clip_probs(x):return np.clip(np.asarray(x,dtype=float),1e-6,1-1e-6)
def logit(x):
    q=clip_probs(x)
    return np.log(q/(1-q)).reshape(-1,1)
def metrics_probability(y,p):
    y=np.asarray(y,dtype=int);p=clip_probs(p)
    assert len(y)==len(p)>0
    indices=np.minimum((p*N_BINS).astype(int),N_BINS-1)
    bins=[];ece=0.
    for k in range(N_BINS):
        ix=np.where(indices==k)[0]
        if len(ix):
            acc=float(y[ix].mean());conf=float(p[ix].mean())
            ece+=len(ix)/len(y)*abs(acc-conf)
        else:acc=conf=float('nan')
        bins.append({'bin':k,'limite_inferior':k/N_BINS,'limite_superior':(k+1)/N_BINS,
                     'n':len(ix),'confianza_media':conf,'precision_empirica':acc,
                     'diferencia_precision_menos_confianza':acc-conf if len(ix) else float('nan')})
    metric={'n_detecciones':len(y),'proporcion_correctas':float(y.mean()),
            'ECE_10_bins':float(ece),'Brier_score':float(brier_score_loss(y,p)),
            'Log_loss':float(log_loss(y,p,labels=[0,1]))}
    metric['ROC_AUC']=float(roc_auc_score(y,p)) if len(set(y))==2 else float('nan')
    metric['Average_precision_detecciones']=float(average_precision_score(y,p)) if len(set(y))==2 else float('nan')
    return metric,pd.DataFrame(bins)

results=[];bin_rows=[];score_rows=[];params=[];models={}
for name,data in labels.items():
    val=data['val'];test=data['test']
    y_val=val['correcta_IoU05'].to_numpy(dtype=int)
    assert set(y_val)=={0,1},f'Validación debe contener aciertos y errores: {name}'
    model=LogisticRegression(C=1.0,solver='lbfgs',max_iter=1000,random_state=SEED)
    model.fit(logit(val['score_raw']),y_val)
    slope=float(model.coef_[0,0]);intercept=float(model.intercept_[0])
    if slope<=0:raise RuntimeError(f'{name}: calibrador no monotónico, revisar puntuaciones de validación antes de continuar.')
    models[name]=(slope,intercept)
    params.append({'modelo':name,'intercepto':intercept,'pendiente_logit':slope,
                   'calibrador':'Platt-logit (LogisticRegression C=1, solo validación)'})
    for part,df in [('validacion',val),('prueba',test)]:
        y=df['correcta_IoU05'].to_numpy(dtype=int)
        raw=clip_probs(df['score_raw'])
        cal=clip_probs(model.predict_proba(logit(df['score_raw']))[:,1])
        for label,p in [('Sin calibrar',raw),('Calibrado',cal)]:
            metric,bins=metrics_probability(y,p)
            results.append({'modelo':name,'particion':part,'estado':label,**metric})
            bins.insert(0,'modelo',name);bins.insert(1,'particion',part);bins.insert(2,'estado',label)
            bin_rows.append(bins)
        small=df[['modelo','particion','imagen_id','imagen','fecha','score_raw','correcta_IoU05']].copy()
        small['score_calibrado']=cal
        score_rows.append(small)
        if part=='prueba':
            for label,p in [('Sin calibrar',raw),('Calibrado',cal)]:
                print(name,label,'test:',metrics_probability(y,p)[0])
df_metrics=pd.DataFrame(results)
df_bins=pd.concat(bin_rows,ignore_index=True)
df_scores=pd.concat(score_rows,ignore_index=True)
df_params=pd.DataFrame(params)


## D. Figura editorial de cuatro paneles, ECE real y barras de brecha

In [ ]:
# Paleta inspirada en la referencia del usuario. Los valores de ECE
# se calculan de datos experimentales, NUNCA se asignan manualmente.
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':10,
    'axes.labelsize':11,'axes.titlesize':11,'axes.linewidth':0.7,
    'pdf.fonttype':42,'ps.fonttype':42,'savefig.facecolor':'white'})
BLUE='#5E88AD';RED='#E98583';GRAY='#35383C'
fig,axes=plt.subplots(2,2,figsize=(12.2,9.3),sharex=True,sharey=True,
                      gridspec_kw={'hspace':0.28,'wspace':0.20})
for i,model in enumerate(['YOLO11n-seg','Mask R-CNN']):
    for j,status in enumerate(['Sin calibrar','Calibrado']):
        ax=axes[i,j]
        b=df_bins[(df_bins['modelo']==model)&(df_bins['particion']=='prueba')&(df_bins['estado']==status)]
        ece=float(df_metrics[(df_metrics['modelo']==model)&(df_metrics['particion']=='prueba')&
                             (df_metrics['estado']==status)]['ECE_10_bins'].iloc[0])
        for row in b.itertuples(index=False):
            if row.n==0:continue
            x=(row.limite_inferior+row.limite_superior)/2
            # Barra azul = tasa observada de predicciones verdaderas en el bin.
            ax.bar(x,row.precision_empirica,width=0.087,color=BLUE,edgecolor='white',linewidth=0.6,zorder=2)
            # Área salmón = discrepancia absoluta respecto a confianza media.
            lo=min(row.precision_empirica,row.confianza_media)
            hi=max(row.precision_empirica,row.confianza_media)
            ax.add_patch(Rectangle((x-0.0435,lo),0.087,hi-lo,
                                   facecolor=RED,edgecolor='none',alpha=0.88,zorder=3))
        ax.plot([0,1],[0,1],linestyle='--',linewidth=1.1,color=GRAY,zorder=5)
        ax.set_title(f'{"(a)" if (i,j)==(0,0) else "(b)" if (i,j)==(0,1) else "(c)" if (i,j)==(1,0) else "(d)"} '
                     f'{model} — {status} (ECE={ece:.4f})',fontweight='bold',pad=10)
        ax.set_xlim(0,1);ax.set_ylim(0,1)
        ax.set_xticks(np.arange(0,1.01,.2));ax.set_yticks(np.arange(0,1.01,.2))
        ax.grid(alpha=.13,zorder=0)
        if i==1:ax.set_xlabel('Prediction confidence')
        if j==0:ax.set_ylabel('Empirical detection precision')
handles=[Line2D([0],[0],ls='--',color=GRAY,label='Perfect calibration'),
         Patch(facecolor=BLUE,label='Observed precision'),Patch(facecolor=RED,label='Calibration gap')]
fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False,
           bbox_to_anchor=(0.5,-0.005),fontsize=10)
fig.suptitle('Confidence reliability of pig instance detections (IoU ≥ 0.50)',fontsize=14,
             weight='bold',y=0.99)
fig.subplots_adjust(top=0.91,bottom=0.105)
FBASE=FIGURES/'CERDOS_14_Diagrama_Confiabilidad'
for ext in ['png','pdf','svg']:
    fig.savefig(str(FBASE)+'.'+ext,bbox_inches='tight',
                dpi=600 if ext=='png' else 300)
plt.show();plt.close(fig)
print('✅ Figuras originales:',str(FBASE)+'.[png,pdf,svg]')


## E. Comparación por fecha, exportaciones y límites metodológicos

In [ ]:
rows=[]
for model in labels:
    d=df_scores[(df_scores['modelo']==model)&(df_scores['particion']=='prueba')]
    for date,g in d.groupby('fecha'):
        for label,col in [('Sin calibrar','score_raw'),('Calibrado','score_calibrado')]:
            metric,_=metrics_probability(g['correcta_IoU05'],g[col])
            rows.append({'modelo':model,'fecha':date,'estado':label,**metric})
df_dates=pd.DataFrame(rows)
limitations=[
 'ECE, Brier y Log loss cuantifican calibración de detecciones propuestas; no de píxeles ni de animales sin detección.',
 'Etiquetas binarias: emparejamiento uno a uno por IoU de máscara ≥ 0,50; detecciones duplicadas son falsas.',
 'Umbral mínimo de inferencia de ambos modelos 0,001; calibración no cambia emparejamiento ni ranking con Platt monotónico.',
 'Los falsos negativos están registrados por separado y no contribuyen al denominador del ECE.',
 'Calibradores ajustados únicamente sobre imágenes de validación de cuatro fechas; no se entrenó ninguna red.',
 'Test de cinco fechas ya consultado en notebooks 04, 07, 09–12: evaluación posterior exploratoria, no test externo nuevo.',
 'AP de segmentación y ECE son métricas distintas; reducción de ECE no equivale a aumento automático del mAP o del conteo.',
 'Dependencias entre imágenes dentro de fecha, misma granja/dominio no garantizados; no se afirma validación externa.',
 'Los intervalos de incertidumbre de ECE requieren análisis agrupado por fecha, que con cinco fechas sería muy impreciso.'
]
protocol={'version':'CERDOS_14_PLATT_DETECCIONES_v1','fecha_utc':datetime.now(timezone.utc).isoformat(),
    'target':'deteccion_correcta_1_1_IoU_mascara_0.50','calibrador':'Platt con logit(score), C=1, datos val',
    'bins_ECE':N_BINS,'umbral_original_inferencia':THRESHOLD_SOURCE,
    'dataset_val_imagenes':196,'dataset_test_imagenes':373,
    'score_eval':'todas_las_detecciones_retenidas_en_09_13',
    'sha_gt_val':sha256(VGT),'sha_gt_test':sha256(TGT),
    'sha_val_preds':{n:sha256(p) for n,p in VP.items()},
    'sha_test_preds':{n:sha256(p) for n,p in TP.items()},
    'pesos_sha256':{'YOLO11n-seg':p13['yolo_sha256'],'Mask R-CNN':p13['mask_sha256']},
    'limitaciones':limitations}
EXCEL=METRICS/'CERDOS_14_Calibracion_Probabilistica.xlsx'
REPORT=REPORTS/'CERDOS_14_Calibracion_Probabilistica.json'
with pd.ExcelWriter(EXCEL,engine='openpyxl') as writer:
    df_metrics.to_excel(writer,sheet_name='Resumen_ECE_Brier',index=False)
    df_bins.to_excel(writer,sheet_name='Bines_Confiabilidad',index=False)
    df_params.to_excel(writer,sheet_name='Platt_Param_Val',index=False)
    df_dates.to_excel(writer,sheet_name='ECE_por_Fecha_Test',index=False)
    df_counts.to_excel(writer,sheet_name='Conteo_GT_TP_FP_FN',index=False)
    df_scores.to_excel(writer,sheet_name='Detecciones_Prob',index=False)
    pd.DataFrame([{'limitacion':s} for s in limitations]).to_excel(writer,sheet_name='Limitaciones',index=False)
REPORT.write_text(json.dumps({'protocolo':protocol,
    'resumen':df_metrics.replace({np.nan:None}).to_dict('records'),
    'parametros_platt':df_params.to_dict('records')},indent=2,ensure_ascii=False),encoding='utf-8')
for p in [EXCEL,REPORT,*[Path(str(FBASE)+'.'+ext) for ext in ['png','pdf','svg']]]:
    assert p.is_file() and p.stat().st_size>0,f'Falta archivo: {p}'
ZIP=EXPORT/'CERDOS_14_Calibracion_Probabilistica_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in [EXCEL,REPORT,*[Path(str(FBASE)+'.'+ext) for ext in ['png','pdf','svg']]]:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
print('✅ Excel:',EXCEL,'\n✅ ZIP:',ZIP)
display(df_metrics[['modelo','particion','estado','n_detecciones','ECE_10_bins','Brier_score','Log_loss']])
colab_files.download(str(ZIP))
